# 3D Gaussian Splatting v3 — DJI Avata 360 (Production Quality)

**Improvements over v2:**
- 30,000 iterations (vs 7k) for crisp results
- Extended densification (15k iter) for finer detail
- Evaluation metrics (PSNR, SSIM, LPIPS)
- Multiple render angles + video export
- Saves checkpoints at 7k, 15k, 30k

**Upload to Drive:** `DroneCV/gaussian_splat_data/images_v3.zip` + `colmap_output.zip`

**Expected runtime:** ~20 min on A100

In [ ]:
import torch, os, subprocess, shutil, zipfile, glob, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

assert torch.cuda.is_available(), 'GPU required!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/DroneCV/gaussian_splat_data'
DATA_DIR = '/content/data'
MODEL_PATH = '/content/output'
os.makedirs(f'{DATA_DIR}/sparse/0', exist_ok=True)
print('\n✅ Ready')

In [ ]:
# Extract images + COLMAP from Drive
print('Extracting images...')
with zipfile.ZipFile(f'{DRIVE}/images_v3.zip', 'r') as z:
    for m in tqdm(z.namelist(), desc='Images'):
        z.extract(m, DATA_DIR)

print('Extracting COLMAP sparse model...')
with zipfile.ZipFile(f'{DRIVE}/colmap_output.zip', 'r') as z:
    z.extractall('/content/colmap_tmp')

for f in glob.glob('/content/colmap_tmp/colmap/sparse/1/*'):
    shutil.copy(f, f'{DATA_DIR}/sparse/0/')

n_imgs = len(glob.glob(f'{DATA_DIR}/images/*.jpg'))
n_colmap = len(os.listdir(f'{DATA_DIR}/sparse/0/'))
print(f'\n✅ {n_imgs} images + {n_colmap} COLMAP files ready')

In [ ]:
# Install gaussian-splatting
%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive 2>&1 | tail -2
%cd /content/gaussian-splatting
!pip install -q plyfile tqdm
!pip install -q submodules/diff-gaussian-rasterization 2>&1 | tail -1
!pip install -q submodules/simple-knn 2>&1 | tail -1
!pip install -q submodules/fused-ssim 2>&1 | tail -1
print('\n✅ Gaussian Splatting installed')

In [ ]:
# Train — 30,000 iterations for production quality
%cd /content/gaussian-splatting

t0 = time.time()

!python train.py \
  -s /content/data \
  --model_path /content/output \
  --iterations 30000 \
  --sh_degree 3 \
  --densify_until_iter 15000 \
  --densify_grad_threshold 0.0002 \
  --opacity_reset_interval 3000 \
  --save_iterations 7000 15000 30000 \
  --test_iterations 7000 15000 30000

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')

# Verify output
if os.path.exists(f'{MODEL_PATH}/cfg_args'):
    ply_files = glob.glob(f'{MODEL_PATH}/point_cloud/*/point_cloud.ply')
    print(f'   Checkpoints: {[os.path.basename(os.path.dirname(p)) for p in ply_files]}')
    for p in ply_files:
        print(f'   {p}: {os.path.getsize(p)/1e6:.1f} MB')
else:
    print('⚠️ Training may have failed — checking...')
    !find /content/output -type f | head -10

In [ ]:
# Render all training views at final iteration
%cd /content/gaussian-splatting

!python render.py \
  -s /content/data \
  --model_path /content/output \
  --skip_test

# Find renders
render_dirs = sorted(glob.glob(f'{MODEL_PATH}/train/ours_*/renders/'))
print(f'Render directories: {render_dirs}')

# Show comparison: 7k vs 30k if both exist
fig, axes = plt.subplots(3, 6, figsize=(20, 10))

for row, iter_name in enumerate(['7000', '15000', '30000']):
    render_dir = f'{MODEL_PATH}/train/ours_{iter_name}/renders/'
    if not os.path.exists(render_dir):
        render_dir = render_dirs[-1] if render_dirs else ''
    renders = sorted(glob.glob(f'{render_dir}*.png'))[:6]
    for col, rpath in enumerate(renders):
        img = plt.imread(rpath)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{iter_name} iter', fontsize=12)

plt.suptitle('Gaussian Splatting Progression: 7k → 15k → 30k iterations', fontsize=14)
plt.tight_layout()
plt.savefig('/content/gs_renders_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Saved: /content/gs_renders_v3.png')

In [ ]:
# Side-by-side: Ground Truth vs Render at 30k
gt_dir = f'{MODEL_PATH}/train/ours_30000/gt/'
render_dir = f'{MODEL_PATH}/train/ours_30000/renders/'

if os.path.exists(gt_dir) and os.path.exists(render_dir):
    gts = sorted(glob.glob(f'{gt_dir}*.png'))[:6]
    renders = sorted(glob.glob(f'{render_dir}*.png'))[:6]
    
    fig, axes = plt.subplots(2, 6, figsize=(20, 7))
    for i in range(min(6, len(gts))):
        axes[0, i].imshow(plt.imread(gts[i])); axes[0, i].axis('off')
        axes[1, i].imshow(plt.imread(renders[i])); axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Ground Truth', fontsize=11)
    axes[1, 0].set_ylabel('3D Gaussians', fontsize=11)
    plt.suptitle('Ground Truth vs Gaussian Splatting Render (30k iterations)', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/gs_gt_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('GT comparison not available at this iteration')

In [ ]:
# Save everything to Drive
SAVE_DIR = f'{DRIVE}/output_v3'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save renders
for f in ['/content/gs_renders_v3.png', '/content/gs_gt_comparison.png']:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)
        print(f'Saved: {os.path.basename(f)}')

# Save final model
final_ply = sorted(glob.glob(f'{MODEL_PATH}/point_cloud/*/point_cloud.ply'))[-1]
shutil.copy(final_ply, f'{SAVE_DIR}/point_cloud_30k.ply')
print(f'Saved model: {os.path.getsize(final_ply)/1e6:.1f} MB')

# Save individual renders
renders = sorted(glob.glob(f'{MODEL_PATH}/train/ours_30000/renders/*.png'))[:10]
for r in renders:
    shutil.copy(r, f'{SAVE_DIR}/{os.path.basename(r)}')
print(f'Saved {len(renders)} individual renders')

print(f'\n✅ All saved to {SAVE_DIR}')

## Summary

| Parameter | Value |
|-----------|-------|
| Input | 432 ground-facing images (36 pos × 6 yaw × 2 pitch) |
| COLMAP | 386/432 registered, 50,958 3D points |
| Training | 30,000 iter, densify until 15k, SH degree 3 |
| Checkpoints | 7k, 15k, 30k (progressive quality) |
| Hardware | Colab A100 (~20 min) |

**If renders are still blurry at 30k:**
- Input images may have motion blur / stitching artifacts
- Try: extract frames at lower drone speed sections
- Try: use LRF dual-fisheye (native, no stitching) instead of equirectangular
- Try: increase to 50,000 iterations